# Optimization Landscape

This notebook is the interactive companion to `notes.md`. It uses small but expressive toy objectives to build the main ideas behind optimization landscapes: saddles, degeneracy, symmetry, sharpness, connectivity, and stability.

**Coverage:** critical points, Hessian spectra, sharpness proxies, parameter-space symmetry, interpolation barriers, mode connectivity, overparameterized minima, and edge-of-stability intuition.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="colorblind")
    HAS_SNS = True
except ImportError:
    plt.style.use("seaborn-v0_8-whitegrid")
    HAS_SNS = False

mpl.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 120,
    "font.size": 13,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "legend.framealpha": 0.85,
    "lines.linewidth": 2.0,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.bbox": "tight",
    "savefig.dpi": 150,
})
np.random.seed(42)
print("Plot setup complete.")


In [ ]:
COLORS = {
    "primary": "#0077BB",
    "secondary": "#EE7733",
    "tertiary": "#009988",
    "error": "#CC3311",
    "neutral": "#555555",
    "highlight": "#EE3377",
}

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_close(name, value, target, tol=1e-8):
    ok = np.allclose(value, target, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    if not ok:
        print("  value :", value)
        print("  target:", target)
    return ok

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

def numerical_hessian(f, theta, eps=1e-5):
    theta = np.array(theta, dtype=float)
    n = theta.size
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            e_i = np.zeros(n)
            e_j = np.zeros(n)
            e_i[i] = eps
            e_j[j] = eps
            H[i, j] = (
                f(theta + e_i + e_j)
                - f(theta + e_i - e_j)
                - f(theta - e_i + e_j)
                + f(theta - e_i - e_j)
            ) / (4 * eps**2)
    return H

def loss_uv(theta):
    u, v = theta
    return 0.5 * (u * v - 1.0) ** 2

def grad_uv(theta):
    u, v = theta
    err = u * v - 1.0
    return err * np.array([v, u], dtype=float)

def hess_uv(theta):
    u, v = theta
    return np.array([
        [v**2, 2 * u * v - 1.0],
        [2 * u * v - 1.0, u**2],
    ], dtype=float)

def scalar_model_outputs(theta, xs):
    u, v = theta
    return (u * v) * xs

def line_path(a, b, ts):
    return np.array([(1 - t) * np.array(a) + t * np.array(b) for t in ts])

def bezier_path(a, c, b, ts):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    c = np.array(c, dtype=float)
    return np.array([(1 - t) ** 2 * a + 2 * t * (1 - t) * c + t ** 2 * b for t in ts])

def path_barrier(path, loss_fn):
    vals = np.array([loss_fn(p) for p in path])
    return vals.max() - max(vals[0], vals[-1]), vals

def loss_abc(theta):
    a, b, c = theta
    return 0.5 * (a * b * c - 1.0) ** 2

def grad_abc(theta):
    a, b, c = theta
    err = a * b * c - 1.0
    return err * np.array([b * c, a * c, a * b], dtype=float)

print("Helpers ready.")


## 1. Local Geometry with a Symmetric Toy Loss

We will use the scalar factorization objective

$$
\mathcal{L}(u,v) = \frac{1}{2}(uv - 1)^2.
$$

It is tiny, but it already contains several key features of deep-learning landscapes:

- nonconvex parameterization
- a strict saddle at the origin
- a full manifold of global minima given by $uv = 1$
- scaling symmetry $(u,v) \mapsto (cu, v/c)$


In [ ]:
header("Critical points of the scalar factorization loss")
saddle = np.array([0.0, 0.0])
minimum_a = np.array([1.0, 1.0])
minimum_b = np.array([2.0, 0.5])

H_saddle = hess_uv(saddle)
H_min_a = hess_uv(minimum_a)
H_min_b = hess_uv(minimum_b)

print("Hessian at saddle:\n", H_saddle)
print("Eigenvalues at saddle:", np.linalg.eigvalsh(H_saddle))
print("Hessian at (1,1):\n", H_min_a)
print("Eigenvalues at (1,1):", np.linalg.eigvalsh(H_min_a))
print("Hessian at (2,0.5):\n", H_min_b)
print("Eigenvalues at (2,0.5):", np.linalg.eigvalsh(H_min_b))

check_true("Origin is a strict saddle", np.min(np.linalg.eigvalsh(H_saddle)) < 0 < np.max(np.linalg.eigvalsh(H_saddle)))
check_close("Loss at both minima is zero", np.array([loss_uv(minimum_a), loss_uv(minimum_b)]), np.zeros(2))


In [ ]:
header("Contour plot of the scalar factorization landscape")
u = np.linspace(-2.5, 2.5, 400)
v = np.linspace(-2.5, 2.5, 400)
U, V = np.meshgrid(u, v)
Z = 0.5 * (U * V - 1.0) ** 2

fig, ax = plt.subplots()
cs = ax.contour(U, V, Z, levels=[0.0, 0.02, 0.1, 0.25, 0.5, 1.0, 2.0], colors=COLORS["primary"])
ax.clabel(cs, inline=True, fontsize=9)
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.scatter([0, 1, 2], [0, 1, 0.5], c=[COLORS["error"], COLORS["tertiary"], COLORS["secondary"]], s=60)
ax.set_title(r"Contours of $\mathcal{L}(u,v)=\frac{1}{2}(uv-1)^2$")
ax.set_xlabel("u")
ax.set_ylabel("v")
plt.show()
check_true("Global minima lie on the hyperbola uv=1", loss_uv(np.array([1.25, 0.8])) < 1e-12)


**What to notice:** the bottom of the landscape is not an isolated point. It is an entire curve. This is the simplest possible illustration of a nonconvex landscape with many equivalent minima.


In [ ]:
header("Sharpness varies along the same zero-loss manifold")
us = np.logspace(-1, 1, 200)
nonzero_eigs = []
for u_val in us:
    H = hess_uv(np.array([u_val, 1.0 / u_val]))
    eigs = np.linalg.eigvalsh(H)
    nonzero_eigs.append(eigs[-1])
nonzero_eigs = np.array(nonzero_eigs)

min_idx = np.argmin(nonzero_eigs)
print("Minimum nonzero eigenvalue occurs near u =", us[min_idx])
print("Smallest nonzero eigenvalue =", nonzero_eigs[min_idx])
print("Eigenvalue at (1,1) =", hess_uv(np.array([1.0, 1.0])).trace())
check_true("Balanced factorization is the least sharp on this manifold", abs(np.log10(us[min_idx])) < 0.1)


In [ ]:
header("Curvature along the manifold uv=1")
fig, ax = plt.subplots()
ax.plot(us, nonzero_eigs, color=COLORS["highlight"])
ax.set_xscale("log")
ax.set_title("Nonzero Hessian eigenvalue along equivalent minima")
ax.set_xlabel("u on the manifold v = 1/u")
ax.set_ylabel("Largest eigenvalue")
plt.show()
check_true("Strongly unbalanced factorizations look sharper", nonzero_eigs[0] > nonzero_eigs[min_idx] and nonzero_eigs[-1] > nonzero_eigs[min_idx])


## 2. Parameter Space vs Function Space

The points $(0.5, 2)$ and $(2, 0.5)$ are far apart in parameter space, but they define the same scalar function $x \mapsto uvx = x$.


In [ ]:
header("Parameter distance can disagree with function distance")
theta_a = np.array([0.5, 2.0])
theta_b = np.array([2.0, 0.5])
xs = np.linspace(-3, 3, 200)

param_distance = np.linalg.norm(theta_a - theta_b)
func_distance = np.mean((scalar_model_outputs(theta_a, xs) - scalar_model_outputs(theta_b, xs)) ** 2)

print("Parameter distance =", param_distance)
print("Function-space MSE =", func_distance)
check_true("The parameter vectors are far apart", param_distance > 2.0)
check_close("The implemented functions are identical", func_distance, 0.0)


In [ ]:
header("Linear interpolation can show a barrier even between equivalent functions")
ts = np.linspace(0, 1, 401)
linear_path = line_path(theta_a, theta_b, ts)
linear_barrier, linear_vals = path_barrier(linear_path, loss_uv)
print("Barrier along the straight line =", linear_barrier)
check_true("The linear path rises above zero loss", linear_barrier > 0.05)


In [ ]:
header("A nonlinear path can connect the same endpoints at zero loss")
us_path = np.exp((1 - ts) * np.log(theta_a[0]) + ts * np.log(theta_b[0]))
nonlinear_path = np.stack([us_path, 1.0 / us_path], axis=1)
nonlinear_barrier, nonlinear_vals = path_barrier(nonlinear_path, loss_uv)
print("Barrier along the hyperbola path =", nonlinear_barrier)
check_close("Nonlinear path stays on the zero-loss manifold", nonlinear_barrier, 0.0, tol=1e-12)


In [ ]:
header("Straight line versus nonlinear low-loss path")
fig, ax = plt.subplots()
ax.plot(ts, linear_vals, label="Linear interpolation", color=COLORS["secondary"])
ax.plot(ts, nonlinear_vals, label="Nonlinear manifold path", color=COLORS["tertiary"])
ax.set_title("Loss along two paths between equivalent solutions")
ax.set_xlabel("t")
ax.set_ylabel("Loss")
ax.legend()
plt.show()
check_true("Mode-connectivity intuition beats straight-line intuition here", linear_vals.max() > nonlinear_vals.max() + 0.05)


## 3. Overparameterization Creates Flat Directions

We now move from two parameters to three:

$$
\mathcal{L}(a,b,c) = \frac{1}{2}(abc - 1)^2.
$$

The global minimum condition $abc = 1$ leaves two free degrees of freedom, so we expect even more degeneracy.


In [ ]:
header("Three-factor model: Hessian rank at a minimum")
theta_star = np.array([1.0, 1.0, 1.0])
H_num = numerical_hessian(loss_abc, theta_star)
eigs = np.linalg.eigvalsh(H_num)
print("Numerical Hessian at (1,1,1):\n", H_num)
print("Eigenvalues:", eigs)
check_true("Two directions are nearly flat", np.sum(np.abs(eigs) < 1e-4) >= 2)


In [ ]:
header("Gradient descent lands at different points on the same manifold")
starts = [
    np.array([1.6, 0.8, 0.7]),
    np.array([0.4, 2.5, 1.2]),
    np.array([2.0, 0.5, 1.5]),
]
finals = []
for start in starts:
    theta = start.copy().astype(float)
    for _ in range(4000):
        theta -= 0.03 * grad_abc(theta)
    finals.append(theta.copy())

finals = np.array(finals)
products = np.prod(finals, axis=1)
print("Final points:\n", finals)
print("Products abc:\n", products)
check_close("All runs end on or near abc=1", products, np.ones_like(products), tol=1e-5)
check_true("Runs end at different parameter vectors", np.linalg.norm(finals[0] - finals[1]) > 0.1)


**What to notice:** overparameterization does not force one unique optimum. It often creates a family of solutions, and optimization chooses one representative from that family.


## 4. Sharpness Proxies and Reparameterization Caveats

A common practical sharpness proxy is $\frac{1}{2}\rho^2 \lambda_{\max}(H)$ for some perturbation radius $\rho$.


In [ ]:
header("Sharpness proxy on balanced vs unbalanced but equivalent minima")
rho = 0.1
theta_bal = np.array([1.0, 1.0])
theta_unbal = np.array([3.0, 1.0 / 3.0])

sharp_bal = 0.5 * rho**2 * np.max(np.linalg.eigvalsh(hess_uv(theta_bal)))
sharp_unbal = 0.5 * rho**2 * np.max(np.linalg.eigvalsh(hess_uv(theta_unbal)))

print("Balanced proxy sharpness   =", sharp_bal)
print("Unbalanced proxy sharpness =", sharp_unbal)
check_true("Equivalent functions can have very different raw sharpness", sharp_unbal > sharp_bal * 2)


In [ ]:
header("Random perturbation loss increase around equivalent minima")
rng = np.random.default_rng(42)
eps = rng.normal(size=(5000, 2))
eps /= np.linalg.norm(eps, axis=1, keepdims=True)
eps *= 0.05

deltas_bal = np.array([loss_uv(theta_bal + e) - loss_uv(theta_bal) for e in eps])
deltas_unbal = np.array([loss_uv(theta_unbal + e) - loss_uv(theta_unbal) for e in eps])

print("Mean perturbation loss increase (balanced)  =", deltas_bal.mean())
print("Mean perturbation loss increase (unbalanced)=", deltas_unbal.mean())
check_true("Unbalanced point is more sensitive in parameter space", deltas_unbal.mean() > deltas_bal.mean())


## 5. Connectivity Beyond Straight Lines

Mode connectivity asks whether two low-loss solutions can be joined by a low-loss curve even when the straight line between them rises.


In [ ]:
header("Quadratic Bezier path search")
best_c = None
best_barrier = np.inf
grid = np.linspace(0.3, 2.2, 120)
for c1 in grid:
    c2 = 1.0 / c1
    candidate = np.array([c1, c2])
    path = bezier_path(theta_a, candidate, theta_b, ts)
    barrier_val, _ = path_barrier(path, loss_uv)
    if barrier_val < best_barrier:
        best_barrier = barrier_val
        best_c = candidate.copy()

print("Best Bezier control point on the manifold:", best_c)
print("Best Bezier barrier:", best_barrier)
check_true("Bezier search reduces the barrier relative to the straight line", best_barrier < linear_barrier)


In [ ]:
header("Line path and Bezier path in parameter space")
bezier = bezier_path(theta_a, best_c, theta_b, ts)
bezier_barrier, bezier_vals = path_barrier(bezier, loss_uv)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].contour(U, V, Z, levels=[0.0, 0.02, 0.1, 0.25, 0.5, 1.0], colors=COLORS["neutral"], alpha=0.6)
axes[0].plot(linear_path[:, 0], linear_path[:, 1], label="Line path", color=COLORS["secondary"])
axes[0].plot(bezier[:, 0], bezier[:, 1], label="Bezier path", color=COLORS["tertiary"])
axes[0].scatter([theta_a[0], theta_b[0], best_c[0]], [theta_a[1], theta_b[1], best_c[1]], c=[COLORS["secondary"], COLORS["secondary"], COLORS["tertiary"]], s=50)
axes[0].set_title("Paths in parameter space")
axes[0].set_xlabel("u")
axes[0].set_ylabel("v")
axes[0].legend()

axes[1].plot(ts, linear_vals, label="Line path", color=COLORS["secondary"])
axes[1].plot(ts, bezier_vals, label="Bezier path", color=COLORS["tertiary"])
axes[1].set_title("Loss along the paths")
axes[1].set_xlabel("t")
axes[1].set_ylabel("Loss")
axes[1].legend()
plt.tight_layout()
plt.show()
check_true("Bezier path has a lower maximum loss", bezier_vals.max() < linear_vals.max())


## 6. Stability, Curvature, and Learning Rate

For a quadratic objective with Hessian eigenvalues $\lambda_i$, classical stability says gradient descent requires $\eta < 2/\lambda_{\max}$.


In [ ]:
header("Quadratic edge-of-stability toy model")
H = np.diag([1.0, 10.0])
def quad_loss(theta):
    return 0.5 * theta @ H @ theta

def quad_grad(theta):
    return H @ theta

theta0 = np.array([1.0, 1.0])
etas = [0.15, 0.19, 0.21]
histories = {}

for eta in etas:
    theta = theta0.copy()
    loss_hist = []
    for _ in range(40):
        loss_hist.append(quad_loss(theta))
        theta = theta - eta * quad_grad(theta)
    histories[eta] = np.array(loss_hist)

for eta in etas:
    print(f"eta={eta:.2f}, final loss={histories[eta][-1]:.6f}")

check_true("Above-boundary step becomes unstable", histories[0.21][-1] > histories[0.19][-1] * 100)


In [ ]:
header("Loss curves near the stability boundary")
fig, ax = plt.subplots()
for eta, vals in histories.items():
    ax.plot(vals, label=rf"$\eta={eta}$")
ax.set_yscale("log")
ax.set_title("Quadratic GD near the stability threshold")
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.legend()
plt.show()
print("Critical quantity eta * lambda_max:")
for eta in etas:
    print(f"  eta={eta:.2f} -> eta*lambda_max={eta * np.max(np.diag(H)):.2f}")
check_true("The threshold is near 2/lambda_max", abs(0.2 * np.max(np.diag(H)) - 2.0) < 1e-12)


## 7. A Small Practical Checkpoint-Averaging Test

Averaging works best when checkpoints already lie in a compatible low-loss corridor. Our toy loss lets us test both success and failure modes.


In [ ]:
header("Averaging close compatible minima can work")
theta_c = np.array([1.0, 1.0])
theta_d = np.array([1.1, 1 / 1.1])
avg_close = 0.5 * (theta_c + theta_d)

print("Loss(theta_c) =", loss_uv(theta_c))
print("Loss(theta_d) =", loss_uv(theta_d))
print("Loss(average close pair) =", loss_uv(avg_close))
check_true("Average of close compatible points stays low loss", loss_uv(avg_close) < 5e-3)


In [ ]:
header("Averaging distant symmetry-related minima can fail")
avg_far = 0.5 * (theta_a + theta_b)
print("Loss(theta_a) =", loss_uv(theta_a))
print("Loss(theta_b) =", loss_uv(theta_b))
print("Loss(average far pair) =", loss_uv(avg_far))
check_true("Average of distant equivalent meaningful points can leave the manifold", loss_uv(avg_far) > 0.05)


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


---


In [ ]:
header("Takeaway summary")
print("1. Saddles and flat directions are central in nonconvex high-dimensional problems.")
print("2. Overparameterization creates families of solutions rather than one isolated optimum.")
print("3. Raw parameter-space sharpness is not invariant to symmetry-preserving reparameterization.")
print("4. Straight-line interpolation can be misleading; nonlinear low-loss paths may exist.")
print("5. Stability depends strongly on local top curvature.")
print("6. Practical averaging succeeds when checkpoints share a compatible low-loss corridor.")
check_true("Notebook reached the final recap cell", True)
